# Lezione 4 — TensorFlow Quantum e altopiani sterili

## Obiettivi di apprendimento

In questa lezione imparerai a:

- Addestrare un circuito parametrizzato con `tfq.layers.PQC` e un ottimizzatore di Keras
- Spiegare il problema degli altopiani sterili (barren plateaus) e perché limita la profondità dei circuiti addestrabili

In [ ]:
!pip install -q cirq tensorflow tensorflow-quantum

## TFQ: il ponte tra Cirq e Keras

TensorFlow Quantum (TFQ) converte un circuito di Cirq in un tensore di TensorFlow con `tfq.convert_to_tensor`, in modo che scorra attraverso una rete Keras come qualsiasi altro dato. Da lì in poi disponi di questi livelli:

- `tfq.layers.PQC` (circuito quantistico parametrizzato): converte un circuito con simboli in un livello Keras addestrabile. Tu indichi quale circuito usare e quale "operatore di lettura" misurare (per esempio, `cirq.Z(qubit)`), e il livello regola gli angoli del circuito durante l'addestramento, esattamente come un livello denso regola i suoi pesi
- `tfq.layers.Expectation` e `tfq.layers.SampledExpectation`: calcolano il valore di attesa di una misura su un circuito. La versione "Sampled" simula il rumore di un hardware reale: ripete il circuito un numero finito di volte (`repetitions`) invece di calcolare la probabilità esatta
- `tfq.differentiators.ParameterShift`: il metodo che TFQ usa per calcolare i gradienti di un circuito quantistico. Non puoi derivare una porta quantistica con le regole usuali del calcolo; lo spostamento di parametro valuta il circuito con il parametro leggermente spostato in su e in giù, e combina i due risultati

Con questi elementi, addestrare un circuito quantistico somiglia ad addestrare una rete classica: definisci un modello Keras, lo compili con una funzione di perdita e un ottimizzatore, e chiami `.fit(...)`.

## Il problema degli altopiani sterili

Quando aumenti il numero di qubit o la profondità di un circuito quantistico parametrizzato inizializzato a caso, accade qualcosa di controintuitivo: il gradiente (il segnale che indica all'ottimizzatore dove muoversi) diventa, in media, praticamente zero su quasi tutto lo spazio dei parametri. Questa situazione si chiama **altopiano sterile**: il paesaggio della funzione di perdita è così piatto che l'ottimizzatore non sa dove andare, come scendere da una montagna immersi in una nebbia fitta su un terreno completamente piano.

Non è un errore nel tuo codice: è una proprietà matematica dei circuiti quantistici casuali e profondi. Più qubit aggiungi, più rapidamente decresce la varianza del gradiente, e questo decadimento è esponenziale. Per questo, in questo corso, lavori con circuiti volutamente poco profondi e con pochi qubit: meno qubit e meno strati ci sono, meno grave è il problema.

## Apprendimento a strati: una via d'uscita dall'altopiano

Una strategia per mitigare gli altopiani sterili è l'**apprendimento a strati**: invece di inizializzare tutto il circuito a caso e addestrare tutti i parametri insieme, addestri prima uno strato poco profondo, lo congeli (o lo lasci vicino ai valori addestrati) e aggiungi gradualmente un nuovo strato. In questo modo il circuito non attraversa mai una fase di profondità elevata con parametri del tutto casuali, che è proprio la combinazione che fa scattare l'altopiano sterile.

Conta anche scegliere con cura cosa misurare: nell'esperimento che vedrai di seguito verificherai che misurare la somma di più operatori Z (uno per qubit) soffre meno di altopiani sterili rispetto a misurare un unico operatore che combina più qubit insieme (per esempio, Z⊗Z).

## Esperimento: come crolla la varianza del gradiente

Il seguente esperimento riproduce il caso classico degli altopiani sterili: genera circuiti casuali di profondità fissa e misura come cade la varianza del gradiente all'aumentare del numero di qubit. Usa pochi qubit e poche ripetizioni in modo che si esegua in pochi minuti; anche così, la tendenza è già visibile.

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def generar_circuito_aleatorio(qubits, profundidad, parametro):
    """Un circuito con una puerta aleatoria distinta en cada qubit y en cada capa,
    entrelazado con puertas CZ. Solo el primer parámetro es simbólico (entrenable);
    el resto son ángulos aleatorios fijos, tal y como exige el experimento original."""
    circuito = cirq.Circuit()
    for qubit in qubits:
        circuito += cirq.ry(np.pi / 4.0)(qubit)

    for capa in range(profundidad):
        for i, qubit in enumerate(qubits):
            es_el_parametro_entrenable = (i == 0 and capa == 0)
            angulo = parametro if es_el_parametro_entrenable else np.random.uniform(0, 2 * np.pi)
            eleccion = np.random.uniform()
            if eleccion > 2 / 3:
                circuito += cirq.rz(angulo)(qubit)
            elif eleccion > 1 / 3:
                circuito += cirq.ry(angulo)(qubit)
            else:
                circuito += cirq.rx(angulo)(qubit)
        for origen, destino in zip(qubits, qubits[1:]):
            circuito += cirq.CZ(origen, destino)
    return circuito

In [ ]:
def varianza_del_gradiente(num_qubits, profundidad, num_circuitos, repeticiones):
    qubits = [cirq.GridQubit(0, i) for i in range(num_qubits)]
    simbolo = sympy.symbols("parametro")
    circuitos = [generar_circuito_aleatorio(qubits, profundidad, simbolo) for _ in range(num_circuitos)]

    operador_lectura = [sum(cirq.Z(q) for q in qubits)]  # suma de Z, uno por qubit
    diferenciador = tfq.differentiators.ParameterShift()
    capa_expectativa = tfq.layers.SampledExpectation(differentiator=diferenciador)

    circuitos_tensor = tfq.convert_to_tensor(circuitos)
    valores = tf.convert_to_tensor(np.random.uniform(0, 2 * np.pi, (num_circuitos, 1)).astype(np.float32))

    with tf.GradientTape() as cinta:
        cinta.watch(valores)
        salida = capa_expectativa(circuitos_tensor, operators=operador_lectura,
                                   repetitions=repeticiones, symbol_names=[simbolo], symbol_values=valores)

    gradientes = cinta.gradient(salida, valores)
    return tf.math.reduce_std(gradientes, axis=0).numpy()[0]

In [ ]:
# Reducido respecto al experimento original para caber en una sesión de clase
num_circuitos = 20
repeticiones = 500
tamanos_qubits = [2, 4, 6, 8]
profundidad_fija = 20

varianzas = [varianza_del_gradiente(n, profundidad_fija, num_circuitos, repeticiones) for n in tamanos_qubits]

plt.plot(tamanos_qubits, varianzas, marker="o")
plt.xlabel("Número de qubits")
plt.ylabel("Varianza del gradiente")
plt.title("La meseta estéril: la varianza cae al añadir qubits")
plt.show()

## Riepilogo dei punti chiave

- TFQ collega Cirq a Keras: `tfq.layers.PQC` converte un circuito con simboli in un livello addestrabile, e `tfq.differentiators.ParameterShift` ne calcola i gradienti
- `tfq.layers.Expectation` calcola il valore di attesa esatto; `tfq.layers.SampledExpectation` lo approssima simulando un numero finito di ripetizioni, come in un hardware reale
- Gli altopiani sterili sono un crollo esponenziale della varianza del gradiente all'aumentare di qubit o profondità in circuiti inizializzati a caso, e fanno sì che l'ottimizzatore non sappia dove andare
- L'apprendimento a strati (addestrare gradualmente, strato dopo strato, invece di tutto il circuito insieme fin dall'inizio) è una strategia per mitigare questo problema
- Per questo, in questo corso, i circuiti che addestri sono volutamente piccoli e poco profondi